# CrewAI: Multi-Agent Orchestration
## A Hands-On Workshop

**CrewAI** is a framework for building teams of AI agents that collaborate to complete complex tasks.
Instead of wiring a state machine by hand (as you would in LangGraph), you describe *who* each agent is, *what* they need to do, and let the framework coordinate execution.

By the end of this workbook you will have built a two-agent research crew from scratch and understand the three core primitives: `Agent`, `Task`, and `Crew`.

In [ ]:
# Install dependencies — only runs on Google Colab.
# Local users: pip install crewai ddgs python-dotenv
import sys

if "google.colab" in sys.modules:
    %pip install -q crewai ddgs python-dotenv

In [ ]:
import os

from dotenv import load_dotenv

# Local: reads OPENAI_API_KEY from .env
# Colab: paste your key when prompted
load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    import getpass

    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

print("API key loaded:", bool(os.getenv("OPENAI_API_KEY")))

---

## What is CrewAI — and how is it different from LangGraph?

| | LangGraph | CrewAI |
|---|---|---|
| **Mental model** | State machine — you wire nodes and edges | Role-based crew — you describe who does what |
| **You define** | Graph structure, state schema, edge conditions | Agents, tasks, and their relationships |
| **Execution** | You control the flow explicitly | CrewAI orchestrates automatically |
| **Best for** | Fine-grained control, complex branching, HITL | Role-based pipelines, parallel crews |

Neither is better — they solve different problems. This workbook focuses on CrewAI's declarative style.

---

## The Three Core Primitives

```
+-----------------------------------------------------+
|  CREW  -- coordinates everything                   |
|  +-------------+   +-----------------------------+  |
|  |  Agent      |   |  Task                       |  |
|  |  role       |-->|  description + expected out |  |
|  |  goal       |   |  assigned agent             |  |
|  |  backstory  |   |  context (optional)         |  |
|  |  tools      |   +-----------------------------+  |
|  +-------------+                                    |
+-----------------------------------------------------+
```

- **`Agent`** — *who* is doing the work (role, goal, backstory, optional tools)
- **`Task`** — *what* needs to be done (description, expected output, assigned agent)
- **`Crew`** — the orchestrator that runs agents through tasks in order

---

## Part 1 — Agents: Defining Who Does the Work

In [ ]:
from crewai import Agent

# The simplest agent -- no tools, just identity.
# role + goal + backstory shape the system prompt the LLM receives.
researcher = Agent(
    role="Researcher",
    goal="Find accurate, recent information about AI agent frameworks",
    backstory=(
        "You are a thorough researcher who identifies reliable sources "
        "and extracts the most relevant facts for each topic."
    ),
    verbose=True,
)

print("Agent created:", researcher.role)

---

## Part 2 — Tools: Giving Agents Abilities

An agent without tools can only reason over its training data.
Tools let agents interact with the world: search the web, read files, call APIs.

CrewAI tools are defined with the `@tool` decorator. The docstring becomes the tool description — the LLM reads it to decide when to call the tool.

In [ ]:
from crewai.tools import tool


@tool("Web Search")
def web_search(query: str) -> str:
    """Search the web for recent information about a topic. Use for current events and facts."""
    from ddgs import DDGS

    results = list(DDGS().text(query, max_results=5))
    return "\n".join(f"- {r['body']}" for r in results) if results else "No results found."


# Quick test -- confirm the tool works before wiring it to an agent
print(web_search.run("CrewAI framework 2025")[:300])

In [ ]:
# Attach the tool to the researcher agent
researcher = Agent(
    role="Researcher",
    goal="Find accurate, recent information about AI agent frameworks",
    backstory=(
        "You are a thorough researcher who identifies reliable sources "
        "and extracts the most relevant facts for each topic."
    ),
    tools=[web_search],
    verbose=True,
)

print("Tools available:", [t.name for t in researcher.tools])

---

## Part 3 — Tasks: Giving Agents Specific Goals

An `Agent` defines *who* someone is. A `Task` defines *what* they need to deliver.

Two fields do most of the work:
- **`description`** — instructions the agent follows (like a brief)
- **`expected_output`** — the format and quality bar for the result

In [ ]:
from crewai import Task

TOPIC = "LangGraph vs CrewAI for production AI agents"

research_task = Task(
    description=f"Research '{TOPIC}'. Gather 5-7 key facts with source references.",
    expected_output="A numbered list of key findings, each with a brief source note.",
    agent=researcher,
)

print("Task defined:", research_task.description[:80])

---

## Part 4 — The Crew: Orchestrating Multiple Agents

A `Crew` sequences agents through tasks. The `context` parameter on a task tells CrewAI
to pass one task's output as input to the next — this is how agents collaborate.

In [ ]:
from crewai import Crew

# Second agent -- no tools needed, works from the researcher's output
writer = Agent(
    role="Writer",
    goal="Turn research findings into a clear, structured report",
    backstory=(
        "You are a concise technical writer who makes complex research "
        "readable for a general developer audience."
    ),
    verbose=True,
)

# context=[research_task] passes the researcher's output into this task
write_task = Task(
    description="Write a 200-word structured report: intro, key points, conclusion.",
    expected_output="A structured 200-word report with clearly labelled sections.",
    agent=writer,
    context=[research_task],
)

crew = Crew(
    agents=[researcher, writer],
    tasks=[research_task, write_task],
    verbose=True,
)

print(f"Crew ready: {len(crew.agents)} agents, {len(crew.tasks)} tasks")

---

## Part 5 — Run the Crew

`crew.kickoff()` starts execution. Tasks run in order. The final task's output is returned.

> **Cost note:** this makes real LLM calls. Expect 2-4 OpenAI API calls total.

In [ ]:
result = crew.kickoff()

print("\n" + "=" * 60)
print("FINAL REPORT")
print("=" * 60)
print(result)

---

## Exercise 1 — Run the Crew on Your Own Topic

Change `MY_TOPIC` below and re-run the crew. Try something you're curious about:
- `"Retrieval-Augmented Generation in 2025"`
- `"Open-source LLMs vs GPT-4"`
- `"Autonomous AI agents in production"`

In [ ]:
MY_TOPIC = "your topic here"  # <- change this

ex1_research = Task(
    description=f"Research '{MY_TOPIC}'. Gather 5-7 key facts with source references.",
    expected_output="A numbered list of key findings, each with a brief source note.",
    agent=researcher,
)
ex1_write = Task(
    description="Write a 200-word structured report: intro, key points, conclusion.",
    expected_output="A structured 200-word report.",
    agent=writer,
    context=[ex1_research],
)
ex1_crew = Crew(agents=[researcher, writer], tasks=[ex1_research, ex1_write], verbose=False)
print(ex1_crew.kickoff())

---

## Exercise 2 — Add a Third Agent: the Editor

Add an `Editor` agent that reviews the writer's report and improves clarity.

Hints:
1. Define an `editor` `Agent` with role `"Editor"` and a suitable goal and backstory
2. Define an `edit_task` with `context=[write_task]` so it receives the writer's draft
3. Add all three to a new `Crew` with three agents and three tasks

In [ ]:
# Exercise 2 starter -- fill in the blanks

editor = Agent(
    role="Editor",
    goal="TODO",
    backstory="TODO",
    verbose=True,
)

edit_task = Task(
    description="TODO",
    expected_output="TODO",
    agent=editor,
    context=[],  # <- what should go here?
)

# Uncomment when ready:
# three_agent_crew = Crew(agents=[researcher, writer, editor], tasks=[research_task, write_task, edit_task], verbose=True)
# print(three_agent_crew.kickoff())

---

## LangGraph vs CrewAI — Same Goal, Different Style

```python
# LangGraph -- you wire the graph manually
workflow = StateGraph(ResearchState)
workflow.add_node("research", research_node)
workflow.add_node("write",    write_node)
workflow.add_edge("research", "write")
workflow.add_edge("write", END)
app = workflow.compile()
app.invoke({"topic": "AI agents"})

# CrewAI -- you describe roles and tasks, the framework orchestrates
crew = Crew(
    agents=[researcher, writer],
    tasks=[research_task, write_task],
)
crew.kickoff()
```

**Use LangGraph when** you need precise control over flow, conditional branching, or human-in-the-loop interrupts.

**Use CrewAI when** you want to define roles and let agents figure out the steps.

---

## What's Next?

- **Example 14** — [SQL Agent](../14-sql-agent/README.md): natural language to SQL with `create_react_agent`
- **Example 16** — RAG Evaluation with RAGAS: measure retrieval quality automatically
- **CrewAI docs** — https://docs.crewai.com: hierarchical crews, memory, async execution

### Key takeaways

| Concept | What to remember |
|---------|------------------|
| `Agent` | Defines identity: role, goal, backstory, tools |
| `Task` | Defines work: description, expected output, context chain |
| `Crew` | Runs agents through tasks in sequence |
| `@tool` | Wraps any Python function into a callable tool |
| `context=` | Passes one task's output into the next — the key to collaboration |